# InsightOS Qwen3 0.6B Final Visualization Specialist Training

This notebook is the fail-closed Kaggle workflow for the approved D2.7 InsightOS data bundle. It does not use the synthetic smoke adapter or any unapproved examples.


In [ ]:
!nvidia-smi
import os
import torch
print({
    'cuda_available': torch.cuda.is_available(),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'cuda_version': torch.version.cuda,
    'torch_version': torch.__version__,
    'fp16_supported': torch.cuda.is_available(),
})


In [ ]:
import hashlib
import json
import os
import zipfile
from pathlib import Path

BUNDLE = Path('/kaggle/input/insightos-qwen3-0-6b-final/insightos_qwen3_0_6b_training_ready_bundle.zip')
EXPECTED_BUNDLE_SHA256 = os.environ.get('INSIGHTOS_EXPECTED_BUNDLE_SHA256', '0d86682a259dc88971cfe5b56ad229e591579947d60b12d66eef527d2a5f106f')
REQUIRED_SPLITS = {'train.jsonl', 'validation.jsonl', 'evaluation.jsonl'}
ALLOWED_TASK_CATEGORIES = {'visual_reconstruction', 'dashboard_layout'}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def zip_integrity_check(path):
    with zipfile.ZipFile(path) as archive:
        bad_entry = archive.testzip()
        if bad_entry is not None:
            raise RuntimeError(f'Corrupt ZIP entry detected: {bad_entry}')
        return sorted(archive.namelist())

if not BUNDLE.exists():
    raise FileNotFoundError(f'Bundle not found: {BUNDLE}')
bundle_sha256 = sha256_file(BUNDLE)
if bundle_sha256 != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(f'Bundle checksum mismatch: expected {EXPECTED_BUNDLE_SHA256}, found {bundle_sha256}')
archive_members = zip_integrity_check(BUNDLE)
print({'bundle_path': str(BUNDLE), 'bundle_sha256': bundle_sha256, 'expected_bundle_sha256': EXPECTED_BUNDLE_SHA256, 'zip_entries': len(archive_members)})


In [ ]:
with zipfile.ZipFile(BUNDLE) as z:
    names = z.namelist()
    manifest_members = [n for n in names if n.endswith('training_data/dataset_manifest.json')]
    approval_members = [n for n in names if n.endswith('approval/final_training_approval_packet.json')]
    if len(manifest_members) != 1:
        raise RuntimeError('Exactly one dataset manifest is required')
    if len(approval_members) != 1:
        raise RuntimeError('Exactly one approval packet is required')
    manifest = json.loads(z.read(manifest_members[0]))
    approval_packet = json.loads(z.read(approval_members[0]))
    if manifest.get('bundle_sha256') and manifest['bundle_sha256'] != EXPECTED_BUNDLE_SHA256:
        raise RuntimeError('Manifest bundle checksum does not match the expected bundle SHA-256')
    if approval_packet.get('bundle_sha256') and approval_packet['bundle_sha256'] != EXPECTED_BUNDLE_SHA256:
        raise RuntimeError('Approval packet bundle checksum does not match the expected bundle SHA-256')
    split_members = {Path(n).name for n in names if n.endswith('.jsonl') and '/training_data/' in n}
    missing_splits = sorted(REQUIRED_SPLITS - split_members)
    if missing_splits:
        raise RuntimeError(f'Missing approved training splits: {missing_splits}')
    if approval_packet.get('human_approved') is not True:
        raise RuntimeError('Training refused: human approval is missing')
    if approval_packet.get('automatic_approval') is not False:
        raise RuntimeError('Training refused: automatic approval is forbidden')
    if manifest.get('readiness_status') != 'TRAINING_READY_APPROVED':
        raise RuntimeError('Training refused: bundle readiness is blocked pending explicit approval')
    approved_training = manifest.get('approved_training_count', 0)
    approved_validation = manifest.get('approved_validation_count', 0)
    approved_evaluation = manifest.get('approved_evaluation_count', 0)
    if approved_training <= 0 or approved_validation <= 0 or approved_evaluation <= 0:
        raise RuntimeError('Training refused: approved training, validation, and evaluation data are required')
    print({
        'readiness_status': manifest.get('readiness_status'),
        'approved_training_count': approved_training,
        'approved_validation_count': approved_validation,
        'approved_evaluation_count': approved_evaluation,
        'bundle_sha256': bundle_sha256,
    })


The approved D2.7 bundle is the only permitted training source. Synthetic smoke examples, chart-selection tasks, business storytelling tasks, and unapproved dashboard-layout examples are excluded from training and must not be used as reason or evidence for approved data.


In [ ]:
import importlib.metadata as metadata
import subprocess
import sys
from pathlib import Path

EXPECTED_TORCH_VERSION = '2.11.0+cu130'
EXPECTED_CUDA_VERSION = '13.0'
if sys.version_info[:2] != (3, 13):
    raise RuntimeError(f'Kaggle Python {sys.version.split()[0]} is unsupported; this dependency set requires Python 3.13.')
if (torch.__version__ != EXPECTED_TORCH_VERSION
        or metadata.version('torch') != EXPECTED_TORCH_VERSION
        or torch.version.cuda != EXPECTED_CUDA_VERSION or not torch.cuda.is_available()):
    raise RuntimeError(
        f'Kaggle runtime mismatch before dependency installation: expected PyTorch {EXPECTED_TORCH_VERSION} '
        f'with CUDA {EXPECTED_CUDA_VERSION}; found torch={torch.__version__}, '
        f'CUDA={torch.version.cuda}, available={torch.cuda.is_available()}. No PyTorch/CUDA replacement is permitted.'
    )

INSTALL_PACKAGE_PINS = [
    'unsloth==2026.11.1',
    'unsloth-zoo==2026.11.1',
    'transformers==5.17.0',
    'trl==1.15.0',
    'peft==0.21.2',
    'datasets==4.8.5',
    'accelerate==1.15.0',
    'bitsandbytes==0.50.2',
    'sentencepiece',
]
PACKAGE_PIN_RATIONALE = {
    'python': 'Python 3.13 is within Unsloth >=3.9,<3.15 and TRL >=3.11 support.',
    'torch': 'Preserve Kaggle 2.11.0+cu130; a pip constraint locks the existing build and torch is not in the install list.',
    'unsloth': '2026.11.1 declares Python <3.15, torch >=2.4,<2.15, Transformers <=5.17, and TRL <=1.15.',
    'unsloth-zoo': '2026.11.1 satisfies Unsloth 2026.11.1 minimum unsloth_zoo>=2026.11.1.',
    'transformers': '5.17.0 is Unsloth 2026.11.1 compatibility ceiling; Kaggle 5.20.2 exceeds that ceiling.',
    'trl': '1.15.0 is Unsloth 2026.11.1 compatibility ceiling and exposes processing_class/SFTConfig APIs.',
    'peft': '0.21.2 is the reported installed release and meets Unsloth >=0.18.0.',
    'datasets': '4.8.5 is the reported installed release and meets Unsloth >=3.4.1,<5.0.0 with excluded releases avoided.',
    'accelerate': '1.15.0 is the reported installed release and meets Unsloth >=0.34.1.',
    'bitsandbytes': '0.50.2 supports CUDA 13.0 Linux sm75 (T4) and PyTorch >=2.4; it avoids Unsloth exclusions.',
    'compatibility_sources': [
        'https://pypi.org/project/unsloth/2026.11.1/',
        'https://pypi.org/project/trl/1.15.0/',
        'https://huggingface.co/docs/bitsandbytes/main/en/installation',
    ],
}
torch_constraint = Path('/tmp/insightos-kaggle-torch-constraint.txt')
torch_constraint.write_text(f'torch=={EXPECTED_TORCH_VERSION}\n', encoding='utf-8')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
     '--constraint', str(torch_constraint), *INSTALL_PACKAGE_PINS],
    check=True,
)
if (torch.__version__ != EXPECTED_TORCH_VERSION
        or metadata.version('torch') != EXPECTED_TORCH_VERSION
        or torch.version.cuda != EXPECTED_CUDA_VERSION):
    raise RuntimeError('Dependency installation changed the preinstalled PyTorch/CUDA runtime; stop and restore Kaggle runtime.')
print({'installed_package_pins': INSTALL_PACKAGE_PINS, 'pytorch_preserved': torch.__version__,
       'cuda_preserved': torch.version.cuda, 'pin_rationale': PACKAGE_PIN_RATIONALE})


In [ ]:
import importlib.metadata as metadata
import sys
from packaging.specifiers import SpecifierSet
from packaging.version import InvalidVersion, Version

STATIC_COMPATIBILITY_MATRIX = {
    'python': '>=3.13,<3.14',
    'torch': '==2.11.0+cu130',
    'transformers': '==5.17.0',
    'datasets': '==4.8.5',
    'trl': '==1.15.0',
    'peft': '==0.21.2',
    'unsloth': '==2026.11.1',
    'unsloth-zoo': '==2026.11.1',
    'bitsandbytes': '==0.50.2',
    'accelerate': '==1.15.0',
}
UNSLOTH_UPSTREAM_SPECIFIERS = {
    'torch': '>=2.4.0,<2.15.0',
    'transformers': '>=4.52.4,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.17.0',
    'trl': '>=0.18.2,!=0.19.0,<=1.15.0',
    'datasets': '>=3.4.1,!=4.0.*,!=4.1.0,!=4.4.*,!=4.5.0,<5.0.0',
    'peft': '>=0.18.0,!=0.11.0',
    'unsloth-zoo': '>=2026.11.1',
    'bitsandbytes': '>=0.45.5,!=0.46.0,!=0.48.0',
    'accelerate': '>=0.34.1',
}

RUNTIME_VERIFIED_VERSIONS = {}
python_version = '.'.join(str(part) for part in sys.version_info[:3])
RUNTIME_VERIFIED_VERSIONS['python'] = {
    'installed': python_version,
    'required': STATIC_COMPATIBILITY_MATRIX['python'],
    'version_ok': Version(python_version) in SpecifierSet(STATIC_COMPATIBILITY_MATRIX['python']),
}
if not RUNTIME_VERIFIED_VERSIONS['python']['version_ok']:
    raise RuntimeError(f'Unsupported Kaggle Python {python_version}; required {STATIC_COMPATIBILITY_MATRIX["python"]}.')

for package, specifier in STATIC_COMPATIBILITY_MATRIX.items():
    if package == 'python':
        continue
    try:
        installed_version = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed_version = None
    version_ok = False
    upstream_version_ok = True
    if installed_version is not None:
        try:
            version_ok = Version(installed_version) in SpecifierSet(specifier)
            upstream_specifier = UNSLOTH_UPSTREAM_SPECIFIERS.get(package)
            if upstream_specifier is not None:
                upstream_version_ok = Version(installed_version) in SpecifierSet(upstream_specifier)
                version_ok = version_ok and upstream_version_ok
        except InvalidVersion as exc:
            raise RuntimeError(f'Invalid installed version for {package}: {installed_version!r}') from exc
    RUNTIME_VERIFIED_VERSIONS[package] = {
        'installed': installed_version,
        'required': specifier,
        'upstream_required': UNSLOTH_UPSTREAM_SPECIFIERS.get(package),
        'upstream_version_ok': upstream_version_ok,
        'version_ok': version_ok,
    }
    print({'package': package, 'required': specifier,
           'installed': installed_version, 'version_ok': version_ok})

dependency_failures = [
    package for package, result in RUNTIME_VERIFIED_VERSIONS.items()
    if not result['version_ok']
]
if dependency_failures:
    raise RuntimeError(f'Kaggle dependency compatibility check failed for: {dependency_failures}')
if torch.__version__ != '2.11.0+cu130' or torch.version.cuda != '13.0' or not torch.cuda.is_available():
    raise RuntimeError(
        f'Kaggle runtime must preserve CUDA-enabled torch 2.11.0+cu130/CUDA 13.0; '
        f'found torch={torch.__version__}, CUDA={torch.version.cuda}, available={torch.cuda.is_available()}.'
    )
DEPENDENCY_VALIDATION_STATUS = 'PASSED'
print({'dependency_validation': DEPENDENCY_VALIDATION_STATUS,
       'static_configuration': STATIC_COMPATIBILITY_MATRIX,
       'kaggle_runtime_verified': RUNTIME_VERIFIED_VERSIONS})


In [ ]:
import json
import zipfile
from collections import defaultdict

from datasets import Dataset

def validate_messages(record, split_name):
    if not isinstance(record, dict):
        raise TypeError(f'{split_name} record is not a JSON object')
    if 'messages' not in record:
        raise ValueError(f'{split_name} record is missing the messages field')
    messages = record.get('messages')
    if not isinstance(messages, list) or len(messages) != 3:
        raise ValueError(f'{split_name} example must contain exactly three messages')
    roles = [message.get('role') for message in messages]
    if roles != ['system', 'user', 'assistant']:
        raise ValueError(f'{split_name} example has invalid role order: {roles!r}')
    for message in messages:
        if not isinstance(message, dict):
            raise TypeError(f'{split_name} message is not a JSON object')
        content = message.get('content')
        if not isinstance(content, str) or not content.strip():
            raise ValueError(f'{split_name} example contains empty or invalid message content')
    return messages

def extract_candidate_identity(record):
    for key in ('candidate_id', 'candidate_key', 'example_id', 'record_id', 'visualization_id', 'layout_id'):
        value = record.get(key)
        if value not in (None, ''):
            return value
    raise ValueError('Record is missing a stable candidate identity')

# Do not treat source-artifact or PBIX identities as unique visual candidate IDs.
# Multiple legitimate examples may originate from the same underlying artifact within one split,
# but a candidate must remain unique across the full approved bundle.

def extract_source_artifact_identity(record):
    for key in ('source_record_id', 'source_artifact_id', 'source_file_id', 'source_file_name', 'source_visual_identity', 'source_id'):
        value = record.get(key)
        if value not in (None, ''):
            return value
    return None

def load_split(name):
    with zipfile.ZipFile(BUNDLE) as z:
        matches = [n for n in z.namelist() if n.endswith(f'/training_data/{name}.jsonl')]
        if len(matches) != 1:
            raise RuntimeError(f'Expected exactly one approved {name} split, found {len(matches)}')
        lines = [line for line in z.read(matches[0]).decode('utf-8').splitlines() if line.strip()]
    rows = []
    seen = set()
    for line in lines:
        record = json.loads(line)
        if not isinstance(record, dict):
            raise TypeError(f'{name} split contains a non-object JSON record')
        validate_messages(record, name)
        task_category = record.get('task_category')
        if task_category not in ALLOWED_TASK_CATEGORIES:
            raise ValueError(f'{name} contains an unsupported task category: {task_category!r}')
        if task_category == 'visual_reconstruction':
            if not record.get('visualization_id') or not record.get('source_visual_identity'):
                raise ValueError(f'{name} visual reconstruction record is missing mandatory upstream evidence')
        elif task_category == 'dashboard_layout':
            if not record.get('layout') or not record.get('page_geometry'):
                raise ValueError(f'{name} dashboard layout record is missing mandatory geometry evidence')
        candidate_key = extract_candidate_identity(record)
        if candidate_key in seen:
            raise ValueError(f'Duplicate candidate identifier detected in {name}: {candidate_key!r}')
        seen.add(candidate_key)
        rows.append(record)
    if not rows:
        raise RuntimeError(f'Approved {name} split is empty; training is blocked until valid approved records exist')
    return rows

train_rows = load_split('train')
validation_rows = load_split('validation')
evaluation_rows = load_split('evaluation')

all_rows_by_split = {
    'train': train_rows,
    'validation': validation_rows,
    'evaluation': evaluation_rows,
}

candidate_seen = {}
artifact_seen = {}
project_seen = {}
cross_split_projects = set()

for split_name, rows in all_rows_by_split.items():
    for record in rows:
        candidate_key = extract_candidate_identity(record)
        if candidate_key in candidate_seen:
            raise RuntimeError(f'Duplicate candidate detected across splits: {candidate_key!r} (first seen in {candidate_seen[candidate_key]}, then in {split_name})')
        candidate_seen[candidate_key] = split_name
        source_identity = extract_source_artifact_identity(record)
        if source_identity is not None:
            if source_identity in artifact_seen and artifact_seen[source_identity] != split_name:
                raise RuntimeError(f'Source artifact identity appears across different splits: {source_identity!r} ({artifact_seen[source_identity]} -> {split_name})')
            artifact_seen[source_identity] = split_name
        project_id = record.get('source_project') or record.get('project_id') or record.get('project')
        if project_id is not None:
            if project_id in project_seen and project_seen[project_id] != split_name:
                cross_split_projects.add(project_id)
            project_seen[project_id] = split_name

if cross_split_projects:
    raise RuntimeError(f'Project-held-out leakage detected across splits: {sorted(cross_split_projects)}')

train_ds = Dataset.from_list(train_rows)
validation_ds = Dataset.from_list(validation_rows)
evaluation_ds = Dataset.from_list(evaluation_rows)

print({
    'train_examples': len(train_ds),
    'validation_examples': len(validation_ds),
    'evaluation_examples': len(evaluation_ds),
    'task_categories': sorted({row.get('task_category') for row in train_rows + validation_rows + evaluation_rows}),
    'project_holdout_check': 'PASSED' if not cross_split_projects else 'FAILED',
})


In [ ]:
import json

CANONICAL_MODEL_ID = 'Qwen/Qwen3-0.6B'
UNSLOTH_MODEL_ID = 'unsloth/Qwen3-0.6B-unsloth-bnb-4bit'
MODEL_ID = CANONICAL_MODEL_ID

MODEL_PROVENANCE = {
    'canonical_base_model': CANONICAL_MODEL_ID,
    'quantized_checkpoint': UNSLOTH_MODEL_ID,
    'runtime_provenance_note': 'The canonical base model is Qwen/Qwen3-0.6B. The Unsloth checkpoint is retained as a separate quantized checkpoint only when runtime Kaggle validation confirms compatibility; it is not silently substituted for the canonical base identity.',
    'preferred_loader': 'Canonical model direct load with compatible Unsloth 4-bit support when available.'
}
print(json.dumps(MODEL_PROVENANCE, indent=2, sort_keys=True))
print({'model_id': MODEL_ID, 'source_base': CANONICAL_MODEL_ID})


In [ ]:
import json
import uuid
from pathlib import Path
import torch
from packaging.specifiers import SpecifierSet
from packaging.version import InvalidVersion, Version

MANUAL_EXECUTION_ENABLED = False
TRAINING_ENABLED = False
RUN_ID = uuid.uuid4().hex
RUN_OUTPUT_DIR = Path('/kaggle/working') / f'insightos_qwen3_0_6b_run_{RUN_ID}'
RUN_STATUS = {
    'base_model_loading': 'NOT_RUN', 'baseline_evaluation': 'NOT_RUN',
    'training_model_loading': 'NOT_RUN', 'qlora_initialization': 'NOT_RUN',
    'training_execution': 'NOT_RUN', 'fine_tuned_evaluation': 'NOT_RUN',
    'adapter_export': 'NOT_RUN', 'artifact_verification': 'NOT_RUN',
}
torch_version = torch.__version__
torch_specifier = STATIC_COMPATIBILITY_MATRIX['torch']
try:
    torch_version_ok = Version(torch_version) in SpecifierSet(torch_specifier)
except InvalidVersion as exc:
    raise RuntimeError(f'Cannot validate installed PyTorch version {torch.__version__!r}: {exc}') from exc
cuda_available = torch.cuda.is_available()
cuda_version = torch.version.cuda
approval_validation_passed = (
    approval_packet.get('human_approved') is True
    and approval_packet.get('automatic_approval') is False
    and manifest.get('readiness_status') == 'TRAINING_READY_APPROVED'
)
bundle_validation_passed = bundle_sha256 == EXPECTED_BUNDLE_SHA256 and bool(archive_members)
dataset_splits_ready = all(len(split) > 0 for split in (train_ds, validation_ds, evaluation_ds))
environment_dependencies_ready = DEPENDENCY_VALIDATION_STATUS == 'PASSED'
approved_run_ready = all((
    approval_validation_passed, bundle_validation_passed,
    dataset_splits_ready, environment_dependencies_ready,
    torch_version_ok, cuda_available, cuda_version == '13.0',
))
BASELINE_GATE_ENABLED = MANUAL_EXECUTION_ENABLED and approved_run_ready
TRAINING_GATE_ENABLED = MANUAL_EXECUTION_ENABLED and TRAINING_ENABLED and approved_run_ready
preflight_report = {
    'run_id': RUN_ID,
    'pytorch': {'installed': torch.__version__, 'required': torch_specifier,
                'version_compatible': torch_version_ok, 'cuda_available': cuda_available,
                'cuda_build_version': cuda_version},
    'environment_readiness': 'NOT_VERIFIED',
    'dependency_versions': RUNTIME_VERIFIED_VERSIONS,
    'dependency_validation': DEPENDENCY_VALIDATION_STATUS,
    'approval_validation': 'PASSED' if approval_validation_passed else 'FAILED',
    'bundle_integrity': 'PASSED' if bundle_validation_passed else 'FAILED',
    'dataset_splits_ready': dataset_splits_ready,
    'manual_execution_enabled': MANUAL_EXECUTION_ENABLED,
    'training_enabled': TRAINING_ENABLED,
    'training_gate_enabled': TRAINING_GATE_ENABLED,
}
preflight_path = Path('/kaggle/working') / f'insightos_preflight_{RUN_ID}.json'
if not torch_version_ok:
    raise RuntimeError(f'Incompatible Kaggle PyTorch {torch.__version__!r}; required {torch_specifier}. This notebook will not install or replace PyTorch; restore a compatible CUDA-enabled Kaggle runtime.')
if not cuda_available or not cuda_version:
    raise RuntimeError(f'CUDA unavailable for installed PyTorch {torch.__version__!r} (torch.version.cuda={cuda_version!r}). Restore Kaggle CUDA-enabled PyTorch; CPU-only installation is prohibited.')
if torch.cuda.device_count() < 1:
    raise RuntimeError('PyTorch reports a CUDA build but no CUDA GPU is visible; Kaggle GPU acceleration is required.')
device_index = 0
device = torch.device(f'cuda:{device_index}')
free_bytes, total_bytes = torch.cuda.mem_get_info(device_index)
gpu_name = torch.cuda.get_device_name(device_index)
print({'cuda_available': cuda_available, 'device_name': gpu_name,
       'free_vram_gib': round(free_bytes / 1024**3, 2),
       'total_vram_gib': round(total_bytes / 1024**3, 2),
       'cuda_version': cuda_version, 'torch_version': torch.__version__, 'fp16_supported': True})
if 'T4' not in gpu_name.upper():
    raise RuntimeError(f'Expected a Kaggle T4 GPU for this configured workflow, detected {gpu_name!r}.')
if free_bytes / 1024**3 < 10:
    raise RuntimeError('Insufficient VRAM for the approved QLoRA workflow. A supported Kaggle T4 or larger GPU is required.')
preflight_report['gpu'] = {
    'name': gpu_name, 'device_count': torch.cuda.device_count(),
    'free_vram_gib': round(free_bytes / 1024**3, 2),
    'total_vram_gib': round(total_bytes / 1024**3, 2),
}
preflight_report['environment_readiness'] = 'PASSED'
preflight_path.write_text(json.dumps(preflight_report, sort_keys=True, indent=2), encoding='utf-8')
print(json.dumps(preflight_report, indent=2, sort_keys=True))
if (MANUAL_EXECUTION_ENABLED or TRAINING_ENABLED) and not approved_run_ready:
    raise RuntimeError('Model execution is blocked: approval, bundle, all dataset splits, dependency compatibility, and CUDA 13.0 must be verified.')
print({'manual_execution_required': True, 'manually_enabled': MANUAL_EXECUTION_ENABLED,
       'training_enabled': TRAINING_ENABLED, 'training_gate_enabled': TRAINING_GATE_ENABLED,
       'preflight_report': str(preflight_path)})


In [ ]:
import json
import math
import gc
import inspect
import torch

from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel
from transformers import TrainingArguments

sft_trainer_parameters = set(inspect.signature(SFTTrainer.__init__).parameters)
sft_config_parameters = set(inspect.signature(SFTConfig).parameters)
training_arguments_parameters = set(inspect.signature(TrainingArguments).parameters)
required_sft_trainer_parameters = {'model', 'args', 'processing_class', 'train_dataset', 'eval_dataset'}
missing_sft_trainer_parameters = required_sft_trainer_parameters - sft_trainer_parameters
if missing_sft_trainer_parameters:
    raise RuntimeError(f'Installed TRL SFTTrainer API is incompatible; missing {sorted(missing_sft_trainer_parameters)}')
if 'max_length' not in sft_config_parameters:
    raise RuntimeError('Installed TRL SFTConfig does not expose max_length required by this notebook.')
if 'eval_strategy' not in sft_config_parameters:
    raise RuntimeError('Installed TRL SFTConfig does not expose Transformers eval_strategy required by this notebook.')
if 'eval_strategy' not in training_arguments_parameters:
    raise RuntimeError('Installed Transformers TrainingArguments does not expose eval_strategy required by this notebook.')
API_COMPATIBILITY_STATUS = 'PASSED'

SCHEMA_REGISTRY_AVAILABLE = False
SCHEMA_VALIDATION_STATUS = "NOT_VERIFIED"

def _is_finite_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)

def _nested_unsupported_properties(expected, generated, path="root"):
    unsupported = []
    if isinstance(expected, dict):
        if not isinstance(generated, dict):
            return False, [f'{path} must be a dictionary']
        for key in generated.keys():
            if key not in expected:
                unsupported.append(f'{path}.{key}')
        for key, expected_value in expected.items():
            if key in generated:
                ok, nested = _nested_unsupported_properties(expected_value, generated[key], f'{path}.{key}')
                if not ok:
                    unsupported.extend(nested)
        return len(unsupported) == 0, unsupported
    if isinstance(expected, list):
        if not isinstance(generated, list):
            return False, [f'{path} must be a list']
        if not expected:
            return True, []
        for index, (expected_item, generated_item) in enumerate(zip(expected, generated)):
            ok, nested = _nested_unsupported_properties(expected_item, generated_item, f'{path}[{index}]')
            if not ok:
                unsupported.extend(nested)
        if len(generated) > len(expected):
            unsupported.append(f'{path} contains extra entries beyond the contract')
        return len(unsupported) == 0, unsupported
    return True, []

def unsupported_property_validation_status():
    if not SCHEMA_REGISTRY_AVAILABLE:
        return 'NOT_VERIFIED'
    return 'NOT_VERIFIED'

def format_example(example):
    messages = example['messages']
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def evaluate_contract(dataset, model, tokenizer):
    if len(dataset) == 0:
        raise RuntimeError('Independent evaluation split is empty')
    visual_rows = [row for row in dataset if row.get('task_category') == 'visual_reconstruction']
    layout_rows = [row for row in dataset if row.get('task_category') == 'dashboard_layout']
    visual_metrics = {
        'samples': len(visual_rows),
        'json_parse_success': 0.0,
        'prescribed_visualization_id_preservation': 0.0,
        'required_field_mapping_correctness': 0.0,
        'axis_and_series_mapping_correctness': 0.0,
        'required_output_properties_present': 0.0,
        'evidence_reference_preservation': 0.0,
        'output_contract_compliance': 'NOT_VERIFIED',
        'unsupported_property_rejection': unsupported_property_validation_status(),
    }
    layout_metrics = {
        'samples': len(layout_rows),
        'json_parse_success': 0.0,
        'canvas_dimensions_validity': 0.0,
        'component_identity_preservation': 0.0,
        'position_coordinate_validity': 0.0,
        'width_height_validity': 0.0,
        'layout_bounds_validity': 0.0,
        'overlap_and_layout_constraints': 'NOT_VERIFIED',
        'layout_contract_compliance': 'NOT_VERIFIED',
        'unsupported_property_rejection': unsupported_property_validation_status(),
    }

    if visual_rows:
        parsed = 0
        id_ok = 0
        field_ok = 0
        axis_ok = 0
        properties_ok = 0
        evidence_ok = 0
        for row in visual_rows:
            messages = row.get('messages')
            if not isinstance(messages, list) or len(messages) != 3:
                continue
            prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
            generated_ids = model.generate(**inputs, max_new_tokens=768, do_sample=False)[0]
            output = tokenizer.decode(generated_ids[inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            try:
                generated = json.loads(output)
            except json.JSONDecodeError:
                continue
            parsed += 1
            expected = json.loads(messages[2]['content'])
            if generated.get('visualization_id') == expected.get('visualization_id'):
                id_ok += 1
            if generated.get('fields') == expected.get('fields'):
                field_ok += 1
            if generated.get('axis') == expected.get('axis') and generated.get('series') == expected.get('series'):
                axis_ok += 1
            required = ['visualization_id', 'fields', 'axis', 'series', 'evidence', 'references']
            if all(key in generated for key in required):
                properties_ok += 1
            if generated.get('evidence') == expected.get('evidence') and generated.get('references') == expected.get('references'):
                evidence_ok += 1
        visual_metrics['json_parse_success'] = parsed / len(visual_rows)
        visual_metrics['prescribed_visualization_id_preservation'] = id_ok / len(visual_rows)
        visual_metrics['required_field_mapping_correctness'] = field_ok / len(visual_rows)
        visual_metrics['axis_and_series_mapping_correctness'] = axis_ok / len(visual_rows)
        visual_metrics['required_output_properties_present'] = properties_ok / len(visual_rows)
        visual_metrics['evidence_reference_preservation'] = evidence_ok / len(visual_rows)

    if layout_rows:
        parsed = 0
        dims_ok = 0
        identity_ok = 0
        position_ok = 0
        size_ok = 0
        bounds_ok = 0
        for row in layout_rows:
            messages = row.get('messages')
            if not isinstance(messages, list) or len(messages) != 3:
                continue
            prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
            generated_ids = model.generate(**inputs, max_new_tokens=768, do_sample=False)[0]
            output = tokenizer.decode(generated_ids[inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            try:
                generated = json.loads(output)
            except json.JSONDecodeError:
                continue
            parsed += 1
            expected = json.loads(messages[2]['content'])
            generated_canvas = generated.get('canvas', {})
            expected_canvas = expected.get('canvas', {})
            canvas_valid = (
                isinstance(generated_canvas, dict)
                and _is_finite_number(generated_canvas.get('width'))
                and _is_finite_number(generated_canvas.get('height'))
                and generated_canvas['width'] > 0 and generated_canvas['height'] > 0
            )
            if canvas_valid:
                dims_ok += 1
            generated_components = generated.get('components', [])
            expected_components = expected.get('components', [])
            generated_by_id = {c.get('id'): c for c in generated_components if isinstance(c, dict) and c.get('id') is not None} if isinstance(generated_components, list) else {}
            expected_by_id = {c.get('id'): c for c in expected_components if isinstance(c, dict) and c.get('id') is not None} if isinstance(expected_components, list) else {}
            unique_ids = (
                isinstance(generated_components, list) and isinstance(expected_components, list)
                and len(generated_by_id) == len(generated_components)
                and len(expected_by_id) == len(expected_components)
            )
            identities_match = unique_ids and generated_by_id.keys() == expected_by_id.keys()
            if identities_match:
                identity_ok += 1
            all_positions_valid = bool(expected_by_id) and identities_match
            all_sizes_valid = all_positions_valid
            all_bounds_valid = all_positions_valid and canvas_valid
            for component_id in (set(generated_by_id) & set(expected_by_id)):
                component = generated_by_id[component_id]
                position = component.get('position', {})
                size = component.get('size', {})
                position_valid = (
                    isinstance(position, dict)
                    and _is_finite_number(position.get('x')) and _is_finite_number(position.get('y'))
                    and position['x'] >= 0 and position['y'] >= 0
                )
                size_valid = (
                    isinstance(size, dict)
                    and _is_finite_number(size.get('width')) and _is_finite_number(size.get('height'))
                    and size['width'] > 0 and size['height'] > 0
                )
                if not position_valid:
                    all_positions_valid = False
                if not size_valid:
                    all_sizes_valid = False
                if not position_valid or not size_valid or not canvas_valid:
                    all_bounds_valid = False
                elif position['x'] + size['width'] > generated_canvas['width'] or position['y'] + size['height'] > generated_canvas['height']:
                    all_bounds_valid = False
            if all_positions_valid:
                position_ok += 1
            if all_sizes_valid:
                size_ok += 1
            if all_bounds_valid:
                bounds_ok += 1
        layout_metrics['json_parse_success'] = parsed / len(layout_rows)
        layout_metrics['canvas_dimensions_validity'] = dims_ok / len(layout_rows)
        layout_metrics['component_identity_preservation'] = identity_ok / len(layout_rows)
        layout_metrics['position_coordinate_validity'] = position_ok / len(layout_rows)
        layout_metrics['width_height_validity'] = size_ok / len(layout_rows)
        layout_metrics['layout_bounds_validity'] = bounds_ok / len(layout_rows)

    return {
        'visual_reconstruction': visual_metrics,
        'dashboard_layout': layout_metrics,
        'schema_validation_status': SCHEMA_VALIDATION_STATUS,
    }

def compare_model_outputs(reference_metrics, tuned_metrics):
    comparison = {}
    for task_name in ['visual_reconstruction', 'dashboard_layout']:
        comparison[task_name] = {}
        for metric_name, tuned_value in tuned_metrics[task_name].items():
            if metric_name in {'samples', 'output_contract_compliance', 'layout_contract_compliance', 'unsupported_property_rejection'}:
                comparison[task_name][metric_name] = 'NOT_VERIFIED'
                continue
            if isinstance(tuned_value, (int, float)) and isinstance(reference_metrics[task_name].get(metric_name), (int, float)):
                comparison[task_name][metric_name] = tuned_value - reference_metrics[task_name][metric_name]
            else:
                comparison[task_name][metric_name] = 'NOT_VERIFIED'
    return comparison

base_metrics = None
if not BASELINE_GATE_ENABLED:
    RUN_STATUS['base_model_loading'] = 'BLOCKED'
    RUN_STATUS['baseline_evaluation'] = 'BLOCKED'
    print({'baseline_evaluation': 'BLOCKED', 'reason': 'An approved manual Kaggle run with verified dependencies, bundle, and non-empty splits is required.'})
else:
    if len(evaluation_rows) == 0:
        raise RuntimeError('Independent evaluation split is empty; evaluation is blocked.')
    RUN_STATUS['base_model_loading'] = 'IN_PROGRESS'
    try:
        base_reference_model, base_reference_tokenizer = FastLanguageModel.from_pretrained(
            model_name=CANONICAL_MODEL_ID, max_seq_length=2048, dtype=torch.float16,
            load_in_4bit=True, device_map={'': device_index},
        )
        RUN_STATUS['base_model_loading'] = 'PASSED'
    except Exception:
        RUN_STATUS['base_model_loading'] = 'FAILED'
        raise
    RUN_STATUS['baseline_evaluation'] = 'IN_PROGRESS'
    try:
        base_metrics = evaluate_contract(evaluation_ds, base_reference_model, base_reference_tokenizer)
        RUN_STATUS['baseline_evaluation'] = 'PASSED'
    except Exception:
        RUN_STATUS['baseline_evaluation'] = 'FAILED'
        raise
    finally:
        if 'base_reference_model' in globals():
            del base_reference_model
        if 'base_reference_tokenizer' in globals():
            del base_reference_tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print({'baseline_evaluation': RUN_STATUS['baseline_evaluation'], 'base_model_identity': CANONICAL_MODEL_ID, 'metrics': base_metrics})


In [ ]:
from unsloth import FastLanguageModel

if not TRAINING_GATE_ENABLED:
    RUN_STATUS['training_model_loading'] = 'BLOCKED'
    print({'training_model_loading': 'BLOCKED', 'reason': 'Both manual execution and training must be enabled after approval and validation.'})
else:
    RUN_STATUS['training_model_loading'] = 'IN_PROGRESS'
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_ID, max_seq_length=2048, dtype=torch.float16,
            load_in_4bit=True, device_map={'': device_index},
        )
        if getattr(model.config, '_name_or_path', MODEL_ID) not in (MODEL_ID, CANONICAL_MODEL_ID):
            raise RuntimeError(f'Loaded model identity does not match canonical {CANONICAL_MODEL_ID!r}')
        RUN_STATUS['training_model_loading'] = 'PASSED'
        print({'training_model_loading': 'PASSED', 'model_id': MODEL_ID,
               'architecture': model.config.model_type,
               'max_position_embeddings': getattr(model.config, 'max_position_embeddings', None)})
    except Exception:
        RUN_STATUS['training_model_loading'] = 'FAILED'
        raise


In [ ]:
from unsloth import FastLanguageModel

if TRAINING_GATE_ENABLED:
    RUN_STATUS['qlora_initialization'] = 'IN_PROGRESS'
    try:
        model = FastLanguageModel.get_peft_model(
            model, r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
            use_gradient_checkpointing='unsloth', random_state=3407,
            target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        )
        RUN_STATUS['qlora_initialization'] = 'PASSED'
        print({'qlora_initialization': 'PASSED',
               'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
               'total_parameters': sum(p.numel() for p in model.parameters())})
    except Exception:
        RUN_STATUS['qlora_initialization'] = 'FAILED'
        raise
else:
    RUN_STATUS['qlora_initialization'] = 'BLOCKED'
    print({'qlora_initialization': 'BLOCKED', 'reason': 'PEFT configuration is not activated until an approved manual Kaggle execution is enabled.'})


In [ ]:
import hashlib
import json
from pathlib import Path
from safetensors import safe_open
from trl import SFTConfig, SFTTrainer

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if not TRAINING_GATE_ENABLED:
    RUN_STATUS['training_execution'] = 'BLOCKED'
    RUN_STATUS['fine_tuned_evaluation'] = 'BLOCKED'
    RUN_STATUS['adapter_export'] = 'BLOCKED'
    RUN_STATUS['artifact_verification'] = 'BLOCKED'
    print({'training_execution': 'BLOCKED', 'reason': 'Both manual Kaggle execution and the explicit training gate must be enabled after approval checks.'})
else:
    RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
    checkpoint_dir = RUN_OUTPUT_DIR / 'checkpoints'
    train_args = SFTConfig(
        output_dir=str(checkpoint_dir), per_device_train_batch_size=1,
        gradient_accumulation_steps=8, learning_rate=2e-4, warmup_ratio=0.03,
        max_steps=-1, num_train_epochs=3, fp16=True, bf16=False,
        logging_steps=10, eval_strategy='steps', eval_steps=25,
        save_strategy='steps', save_steps=25, save_total_limit=3,
        seed=3407, report_to='none', max_length=2048,
    )
    trainer = SFTTrainer(
        model=model, args=train_args, processing_class=tokenizer,
        train_dataset=train_ds, eval_dataset=validation_ds,
    )
    RUN_STATUS['training_execution'] = 'IN_PROGRESS'
    try:
        training_result = trainer.train()
        RUN_STATUS['training_execution'] = 'PASSED'
    except Exception:
        RUN_STATUS['training_execution'] = 'FAILED'
        raise
    RUN_STATUS['fine_tuned_evaluation'] = 'IN_PROGRESS'
    try:
        tuned_metrics = evaluate_contract(evaluation_ds, model, tokenizer)
        RUN_STATUS['fine_tuned_evaluation'] = 'PASSED'
    except Exception:
        RUN_STATUS['fine_tuned_evaluation'] = 'FAILED'
        raise
    adapter_dir = RUN_OUTPUT_DIR / 'adapter'
    RUN_STATUS['adapter_export'] = 'IN_PROGRESS'
    try:
        model.save_pretrained(adapter_dir, safe_serialization=True)
        tokenizer.save_pretrained(adapter_dir)
        RUN_STATUS['adapter_export'] = 'PASSED'
    except Exception:
        RUN_STATUS['adapter_export'] = 'FAILED'
        raise
    RUN_STATUS['artifact_verification'] = 'IN_PROGRESS'
    try:
        adapter_config = adapter_dir / 'adapter_config.json'
        adapter_weights = adapter_dir / 'adapter_model.safetensors'
        if not adapter_config.is_file() or not adapter_weights.is_file():
            raise RuntimeError('Adapter export incomplete: adapter_config.json and adapter_model.safetensors are required.')
        adapter_config_data = json.loads(adapter_config.read_text(encoding='utf-8'))
        if adapter_config_data.get('base_model_name_or_path') != CANONICAL_MODEL_ID:
            raise RuntimeError('Exported adapter configuration does not identify the canonical Qwen3-0.6B base model.')
        with safe_open(str(adapter_weights), framework='pt', device='cpu') as adapter_file:
            if not adapter_file.keys():
                raise RuntimeError('Exported safetensors adapter has no tensors.')
        initial_file_hashes = {
            path.name: {'sha256': sha256_file(path), 'bytes': path.stat().st_size}
            for path in (adapter_config, adapter_weights)
        }
        if any(item['bytes'] <= 0 for item in initial_file_hashes.values()):
            raise RuntimeError('Adapter export contains an empty required artifact.')

        del trainer, model, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        reloaded_model, reloaded_tokenizer = FastLanguageModel.from_pretrained(
            model_name=str(adapter_dir), max_seq_length=2048, dtype=torch.float16,
            load_in_4bit=True, device_map={'': device_index},
        )
        FastLanguageModel.for_inference(reloaded_model)
        inference_messages = evaluation_rows[0]['messages']
        inference_prompt = reloaded_tokenizer.apply_chat_template(
            inference_messages[:2], tokenize=False, add_generation_prompt=True,
        )
        inference_inputs = reloaded_tokenizer(inference_prompt, return_tensors='pt').to(reloaded_model.device)
        with torch.inference_mode():
            inference_output = reloaded_model.generate(**inference_inputs, max_new_tokens=8, do_sample=False)
        generated_token_count = inference_output.shape[-1] - inference_inputs['input_ids'].shape[-1]
        if generated_token_count <= 0:
            raise RuntimeError('Reloaded adapter inference returned no generated tokens.')
        del reloaded_model, reloaded_tokenizer, inference_inputs, inference_output
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        verified_file_hashes = {
            path.name: {'sha256': sha256_file(path), 'bytes': path.stat().st_size}
            for path in (adapter_config, adapter_weights)
        }
        if verified_file_hashes != initial_file_hashes:
            raise RuntimeError('Adapter artifact checksums changed during reload/inference verification.')
        artifact_manifest = {
            'schema_version': '1.0', 'run_id': RUN_ID, 'base_model_id': CANONICAL_MODEL_ID,
            'pytorch': torch.__version__, 'cuda': torch.version.cuda,
            'adapter_directory': str(adapter_dir),
            'files': verified_file_hashes,
            'reload_inference_validation': {'status': 'PASSED', 'generated_tokens': generated_token_count},
        }
        artifact_manifest_path = adapter_dir / 'adapter_export_manifest.json'
        artifact_manifest_path.write_text(json.dumps(artifact_manifest, sort_keys=True, indent=2), encoding='utf-8')
        RUN_STATUS['artifact_verification'] = 'PASSED'
    except Exception:
        RUN_STATUS['artifact_verification'] = 'FAILED'
        raise
    completed_report = {
        'run_id': RUN_ID, 'base_model_identity': CANONICAL_MODEL_ID,
        'adapter_identity': str(adapter_dir), 'run_status': RUN_STATUS,
        'training_metrics': training_result.metrics, 'base_metrics': base_metrics,
        'fine_tuned_metrics': tuned_metrics,
        'metric_deltas': compare_model_outputs(base_metrics, tuned_metrics),
        'adapter_export_manifest': str(artifact_manifest_path), 'model_provenance': MODEL_PROVENANCE,
    }
    (RUN_OUTPUT_DIR / 'completed_training_metrics.json').write_text(json.dumps(completed_report, sort_keys=True, indent=2), encoding='utf-8')
    print(json.dumps(completed_report, indent=2, sort_keys=True))


In [ ]:
import json

approval_status = 'NOT_VERIFIED'
if 'approval_packet' in globals() and 'manifest' in globals():
    approval_status = 'PASSED' if (
        approval_packet.get('human_approved') is True
        and approval_packet.get('automatic_approval') is False
        and manifest.get('readiness_status') == 'TRAINING_READY_APPROVED'
    ) else 'FAILED'
bundle_status = 'NOT_VERIFIED'
if 'bundle_sha256' in globals() and 'archive_members' in globals():
    bundle_status = 'PASSED' if bundle_sha256 == EXPECTED_BUNDLE_SHA256 else 'FAILED'
run_report = {
    'schema_version': '1.0', 'run_id': RUN_ID, 'run_type': 'training_run',
    'preflight_report': str(preflight_path),
    'environment_preflight': preflight_report['environment_readiness'],
    'dependency_validation': DEPENDENCY_VALIDATION_STATUS,
    'api_compatibility_validation': API_COMPATIBILITY_STATUS,
    'bundle_integrity': bundle_status, 'approval_validation': approval_status,
    'dataset_eligibility': 'PASSED' if dataset_splits_ready else 'BLOCKED',
    'model_identity': CANONICAL_MODEL_ID, 'run_status': dict(RUN_STATUS),
    'training_metrics': training_result.metrics if 'training_result' in globals() else None,
    'base_metrics': base_metrics if 'base_metrics' in globals() else None,
    'fine_tuned_metrics': tuned_metrics if 'tuned_metrics' in globals() else None,
    'adapter_export_manifest': str(artifact_manifest_path) if 'artifact_manifest_path' in globals() else None,
    'actual_training_run': TRAINING_GATE_ENABLED and RUN_STATUS['training_execution'] == 'PASSED',
    'automatic_approval_forbidden': True, 'synthetic_data_used': False,
    'model_provenance': MODEL_PROVENANCE,
}
run_report_path = RUN_OUTPUT_DIR / 'training_run_report.json'
if not run_report_path.exists():
    RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    run_report_path.write_text(json.dumps(run_report, sort_keys=True, indent=2), encoding='utf-8')
print(json.dumps(run_report, indent=2, sort_keys=True))
print({'training_run_report': str(run_report_path)})
